In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
REALWORLD2016 Runner (no disk writes)
- Fixed configuration: win=10s | patches=10 | patch_len=50 | sensors=acc+gyr+mag | body=shin
- Five-fold subject-wise CV with 7:1:2 split
- Rich logs; everything in-memory only
"""

import os, math, random, warnings
warnings.filterwarnings("ignore")

from zipfile import ZipFile
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix

from dataclasses import dataclass

@dataclass
class Config:
    # Data
    DATA_PATH: str = "/mnt/share/ali/realworld2016_dataset/"
    LABEL_COL: str = "activity_label"
    SUBJECT_COL: str = "subject_id"

    # Windowing (non-overlapping)
    SIGNAL_RATE: int = 50      # REALWORLD2016 ~50 Hz
    WINDOW_SIZE: int = 500     # 10s windows at 50Hz = 500 samples
    STRIDE: int = 500          # non-overlapping
    PATCH_LEN: int = 50        # 10 patches of 50 samples each for 10s window

    # PatchTST-like
    D_MODEL: int = 56
    N_HEADS: int = 4
    N_LAYERS: int = 4
    DROPOUT: float = 0.3

    # Per-location feature dims (fixed)
    N_STAT_PER_LOC: int = 56
    N_TOPO_PER_LOC: int = 24

    # Training
    BATCH_SIZE: int = 64
    EPOCHS: int = 120
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    MAX_GRAD_NORM: float = 1.0
    EARLY_STOP_PATIENCE: int = 30

    # Repro
    SEED: int = 42

cfg = Config()

# ---------------------------
# 1) LOAD DATA  →  2) WINDOW
# ---------------------------

dataset_path = cfg.DATA_PATH

ALL_BODY_PARTS = [
    "chest", "forearm", "head", "shin", "thigh", "upperarm", "waist",
]

# Sensor types and their file naming patterns
sensor_config = {
    "acc": {"prefix": "acc",           "suffix": ""},
    "gyr": {"prefix": "Gyroscope",     "suffix": ""},
    "mag": {"prefix": "MagneticField", "suffix": ""}
}

# Activity labels mapping
activity_labels = {
    "climbingdown": 0, "climbingup": 1, "jumping": 2, "lying": 3,
    "running": 4, "sitting": 5, "standing": 6, "walking": 7
}
classes = [k for k,_ in sorted(activity_labels.items(), key=lambda x: x[1])]
num_classes = len(classes)

# Sampling
SAMPLING_RATE = cfg.SIGNAL_RATE

def cal_min_time_len(readMe):
    time_len_list = []
    for line in readMe.readlines():
        if line.startswith(b"> entries"):
            try:
                time_len = int(line.split()[2])
                time_len_list.append(time_len)
            except:
                pass
    return min(time_len_list) if time_len_list else 1000

def cal_sensor_data_from_zip_files(sensor_zip_files, sensor_type, label, b_part, min_time_len):
    zip_file = sensor_zip_files.get(sensor_type)
    if not zip_file:
        return np.full([min_time_len, 3], np.nan)
    config = sensor_config[sensor_type]
    csv_file_name = "{}_{}_{}.csv".format(config['prefix'], label, b_part)
    if csv_file_name in zip_file.namelist():
        try:
            f = zip_file.open(csv_file_name)
            df = pd.read_csv(f)
            x = np.array(df["attr_x"]).reshape(-1, 1)
            y = np.array(df["attr_y"]).reshape(-1, 1)
            z = np.array(df["attr_z"]).reshape(-1, 1)
            xyz = np.concatenate((x, y, z), axis=1)
            xyz = xyz[:min_time_len, :]
            f.close()
            return xyz
        except Exception:
            pass
    return np.full([min_time_len, 3], np.nan)

def cal_all_sensor_data(sensor_zip_files, sub_id, label, use_sensors=("acc","gyr","mag"), body_parts=["shin"]):
    # Determine a consistent minimal length across available streams
    min_time_len = 1000
    valid_readme_found = False
    for st in use_sensors:
        zf = sensor_zip_files.get(st)
        if zf:
            try:
                readme_file = zf.open("readMe")
                min_time_len = cal_min_time_len(readme_file)
                readme_file.close()
                if min_time_len > 0:
                    valid_readme_found = True
                    break
            except:
                continue
    if not valid_readme_found or min_time_len <= 0:
        min_time_len = 1000

    all_sensor_data = None
    for sensor_type in use_sensors:
        sensor_data = None
        for b_part in body_parts:
            xyz = cal_sensor_data_from_zip_files(sensor_zip_files, sensor_type, label, b_part, min_time_len)
            if sensor_data is None:
                sensor_data = xyz
            else:
                if xyz.shape[0] != sensor_data.shape[0]:
                    m = min(xyz.shape[0], sensor_data.shape[0])
                    sensor_data = sensor_data[:m, :]
                    xyz = xyz[:m, :]
                sensor_data = np.concatenate((sensor_data, xyz), axis=1)
        if all_sensor_data is None:
            all_sensor_data = sensor_data
        else:
            if sensor_data is not None and sensor_data.shape[0] != all_sensor_data.shape[0]:
                m = min(sensor_data.shape[0], all_sensor_data.shape[0])
                all_sensor_data = all_sensor_data[:m, :]
                sensor_data = sensor_data[:m, :]
            if sensor_data is not None:
                all_sensor_data = np.concatenate((all_sensor_data, sensor_data), axis=1)
    return all_sensor_data

def check_window_quality(window, max_nan_ratio=0.1, max_zero_ratio=0.9):
    total_elements = window.size
    nan_count = np.isnan(window).sum()
    zero_count = (window == 0).sum()
    nan_ratio = nan_count / max(total_elements, 1)
    zero_ratio = zero_count / max(total_elements, 1)
    if nan_ratio > max_nan_ratio: return False
    if zero_ratio > max_zero_ratio: return False
    unique_values_per_channel = [len(np.unique(channel[~np.isnan(channel)])) for channel in window.T]
    constant_channels = sum(1 for count in unique_values_per_channel if count <= 1)
    if constant_channels > 10: return False
    finite_data = window[np.isfinite(window)]
    if finite_data.size:
        max_val = np.max(finite_data)
        min_val = np.min(finite_data)
        if max_val > 10000 or min_val < -10000: return False
    return True

def create_windows_from_data(data, window_size, overlap=0.5, quality_check=True):
    if data is None or len(data) < window_size: return []
    stride = max(1, int(window_size * (1 - overlap)))
    good_windows = []
    for i in range(0, len(data) - window_size + 1, stride):
        window = data[i:i + window_size]
        if quality_check:
            if check_window_quality(window): good_windows.append(window)
        else:
            good_windows.append(window)
    return good_windows

def load_and_window_subject_activity(sub_id, label, window_size_seconds=10, quality_check=True,
                                     use_sensors=("acc","gyr","mag"), body_parts=["shin"], overlap=0.5):
    ziplabel = label.replace("_", "")
    sensor_zip_files = {}
    for sensor_type in ["acc", "gyr", "mag"]:
        if sensor_type not in use_sensors:
            sensor_zip_files[sensor_type] = None
            continue
        zip_file_path = os.path.join(dataset_path, f"proband{sub_id}", "data", f"{sensor_type}_{ziplabel}_csv.zip")
        if os.path.exists(zip_file_path):
            try:
                sensor_zip_files[sensor_type] = ZipFile(zip_file_path)
            except Exception:
                sensor_zip_files[sensor_type] = None
        else:
            sensor_zip_files[sensor_type] = None

    all_data = cal_all_sensor_data(sensor_zip_files, sub_id, ziplabel, use_sensors=use_sensors, body_parts=body_parts)
    for zip_file in sensor_zip_files.values():
        if zip_file: zip_file.close()

    if all_data is not None and len(all_data) > 0:
        window_size_samples = int(SAMPLING_RATE * window_size_seconds)
        return create_windows_from_data(all_data, window_size=window_size_samples, overlap=overlap, quality_check=quality_check)
    else:
        return []

def generate_all_windows(subjects=None, activities=None, window_size_seconds=10, quality_check=True,
                         overlap=0.5, use_sensors=("acc","gyr","mag"), body_parts=["shin"]):
    if subjects is None:
        subjects = list(range(1, 16))  # 1-15
    if activities is None:
        activities = list(activity_labels.keys())
    all_windows, all_labels, all_subjects = [], [], []
    for sub_id in subjects:
        for activity in activities:
            windows = load_and_window_subject_activity(
                sub_id=sub_id, label=activity, window_size_seconds=window_size_seconds,
                quality_check=quality_check, use_sensors=use_sensors, body_parts=body_parts, overlap=overlap
            )
            if windows:
                all_windows.extend(windows)
                label_id = activity_labels[activity]
                all_labels.extend([label_id] * len(windows))
                all_subjects.extend([sub_id] * len(windows))
    return np.array(all_windows, dtype=np.float32), np.array(all_labels, dtype=np.int64), np.array(all_subjects, dtype=np.int64)

# ---------------------------
# 3) MODEL + TRAIN/EVAL (no time embeddings)
# ---------------------------

# Reproducibility
SEED = cfg.SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
if GPU: torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if GPU else "cpu")

def amp_autocast():
    if GPU:
        try:
            return torch.autocast(device_type="cuda", dtype=torch.float16)
        except Exception:
            from torch.cuda.amp import autocast
            return autocast()
    from contextlib import nullcontext
    return nullcontext()

# ====== Feature helpers (stats/topo) ======
try:
    from scipy.signal import find_peaks, peak_prominences
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

def _safe_corr(a, b):
    sa, sb = float(np.std(a)), float(np.std(b))
    if sa < 1e-8 or sb < 1e-8: return 0.0
    c = float(np.corrcoef(a, b)[0, 1]); 
    return 0.0 if not np.isfinite(c) else c

def _mad(x): 
    med = float(np.median(x)); 
    return float(np.median(np.abs(x - med)))

def _skew(x):
    mu, sd = float(np.mean(x)), float(np.std(x))
    if sd < 1e-12: return 0.0
    m3 = float(np.mean((x - mu)**3)); return m3 / (sd**3)

def _kurtosis_excess(x):
    mu, sd = float(np.mean(x)), float(np.std(x))
    if sd < 1e-12: return 0.0
    m4 = float(np.mean((x - mu)**4)); return m4 / (sd**4) - 3.0

def _range(x): return float(np.max(x) - np.min(x)) if x.size else 0.0

def _normalized_psd(x, sr):
    Xc = np.fft.rfft(x)
    ps = (np.abs(Xc) ** 2).astype(np.float64)
    s = ps.sum()
    if s > 0: ps /= s
    freqs_hz = np.fft.rfftfreq(x.shape[0], d=1.0 / sr)
    return ps, freqs_hz

def _spectral_entropy(p):
    p = p[np.isfinite(p) & (p > 0)]
    if p.size == 0: return 0.0
    H = -float(np.sum(p * np.log(p)))
    Hmax = math.log(len(p))
    return float(H / Hmax) if Hmax > 0 else 0.0

def _dominant_two(probs, freqs):
    if probs.size <= 2: return 0.0, 0.0, 0.0, 0.0
    p = probs.copy(); f = freqs.copy()
    p[0] = 0.0
    i1 = int(np.argmax(p)); p1 = float(p[i1]); f1 = float(f[i1])
    p[i1] = -1.0
    i2 = int(np.argmax(p)); p2 = max(float(p[i2]), 0.0); f2 = float(f[i2])
    return f1, p1, f2, p2

def _one_sec_autocorr(x, sr):
    L = min(sr, x.shape[0]-1)
    if L <= 1: return 0.0
    a, b = x[:-L], x[L:]
    return _safe_corr(a, b)

def _lowpass_gravity(a, sr, fc=0.5):
    alpha = math.exp(-2.0 * math.pi * fc / float(sr))
    g = np.zeros_like(a, dtype=np.float32); g[0] = a[0]
    for t in range(1, a.shape[0]): g[t] = alpha * g[t-1] + (1.0 - alpha) * a[t]
    return g

def _angles_from_vec(vx, vy, vz):
    roll = np.arctan2(vy, vz)
    pitch = np.arctan2(-vx, np.sqrt(vy*vy + vz*vz) + 1e-8)
    yaw_p = np.arctan2(vx, vy)
    return roll, pitch, yaw_p

def compute_statistical_features(window_norm: np.ndarray, sr: int = 50) -> np.ndarray:
    # Use first 3 channels as a triad proxy (works for accel-like features)
    tri = window_norm[:, :3]
    x, y, z = tri[:,0], tri[:,1], tri[:,2]
    mag = np.linalg.norm(tri, axis=1)
    feats = []

    # (A) means/stds/ranges
    for sig in (x, y, z):
        feats.extend([float(np.mean(sig)), float(np.std(sig)), _range(sig)])
    feats.append(_safe_corr(x, y)); feats.append(_safe_corr(x, z)); feats.append(_safe_corr(y, z))
    feats.extend([
        float(np.mean(mag)), float(np.std(mag)), _range(mag), _mad(mag),
        _kurtosis_excess(mag), _skew(mag), float(np.median(mag)),
    ])

    # (B) quantiles ×4 signals
    def qpack(sig):
        q25, q50, q75 = np.percentile(sig, [25, 50, 75])
        return [float(np.min(sig)), float(np.max(sig)), float(q50), float(q25), float(q75)]
    for sig in (x, y, z, mag):
        feats.extend(qpack(sig))

    # (C) 1s autocorr
    feats.append(_one_sec_autocorr(mag, sr))

    # (D) spectral
    ps, fq = _normalized_psd(mag, sr)
    f1, p1, f2, p2 = _dominant_two(ps, fq)
    feats.extend([f1, p1, f2, p2])
    feats.append(_spectral_entropy(ps))

    # (E) peaks
    if _HAVE_SCIPY:
        peaks, _ = find_peaks(mag)
        if peaks.size:
            prom = peak_prominences(mag, peaks)[0]
            feats.extend([float(len(peaks)), float(np.median(prom))])
        else:
            feats.extend([0.0, 0.0])
    else:
        if mag.size >= 3:
            pk = ((mag[1:-1] > mag[:-2]) & (mag[1:-1] > mag[2:])).sum()
            feats.append(float(pk))
        else:
            feats.append(0.0)
        feats.append(0.0)

    # (F) angular features
    g = _lowpass_gravity(tri, sr=sr, fc=0.5)
    d = tri - g
    gr, gp, gy = _angles_from_vec(g[:,0], g[:,1], g[:,2]); feats.extend([float(np.mean(gr)), float(np.mean(gp)), float(np.mean(gy))])
    dr, dp, dy = _angles_from_vec(d[:,0], d[:,1], d[:,2])
    for arr in (dr, dp, dy):
        feats.append(float(np.mean(arr))); feats.append(float(np.std(arr)))

    feats = [0.0 if (not np.isfinite(v)) else float(v) for v in feats]
    return np.array(feats, dtype=np.float32)

# ----- Topology (persistent homology) -----
try:
    from ripser import ripser
    _HAVE_RIPSER = True
except Exception:
    _HAVE_RIPSER = False

def _takens_embed(x: np.ndarray, m: int, tau: int) -> np.ndarray:
    T = x.shape[0]
    L = T - (m - 1) * tau
    if L <= 5: return np.zeros((5, m), dtype=np.float32)
    out = np.stack([x[i:i+L] for i in range(0, m*tau, tau)], axis=1)
    return out.astype(np.float32)

def _persistence_entropy(diag: np.ndarray) -> float:
    if diag.size == 0: return 0.0
    births, deaths = diag[:,0], diag[:,1]
    finite = np.isfinite(deaths)
    births, deaths = births[finite], deaths[finite]
    pers = np.maximum(deaths - births, 0.0)
    s = pers.sum()
    if s <= 0: return 0.0
    p = pers / s
    H = -np.sum(p * np.log(p + 1e-12))
    Hmax = np.log(len(p))
    return float(H / (Hmax + 1e-12))

def _topk_lifetimes(diag: np.ndarray, k: int = 3) -> list[float]:
    if diag.size == 0: return [0.0]*k
    births, deaths = diag[:,0], diag[:,1]
    finite = np.isfinite(deaths)
    lifetimes = np.maximum(deaths[finite] - births[finite], 0.0)
    lifetimes = np.sort(lifetimes)[::-1]
    out = np.zeros(k, dtype=np.float32)
    out[:min(k, lifetimes.size)] = lifetimes[:k]
    return out.tolist()

def compute_topological_features(window_norm: np.ndarray, sr: int = 50, m: int = 3, tau: int = 5,
                                 topo_max_points: int = 600) -> np.ndarray:
    tri = window_norm[:, :3]
    mag = np.linalg.norm(tri, axis=1).astype(np.float32)
    if mag.shape[0] > topo_max_points:
        step = int(np.ceil(mag.shape[0] / topo_max_points))
        mag = mag[::step]
    Xemb = _takens_embed(mag, m=m, tau=tau)

    if _HAVE_RIPSER and Xemb.shape[0] >= 8:
        res = ripser(Xemb, maxdim=1)
        D0 = res.get('dgms', [np.empty((0,2)), np.empty((0,2))])[0]
        D1 = res.get('dgms', [np.empty((0,2)), np.empty((0,2))])[1]
    else:
        # Simple recurrence proxy if ripser unavailable
        d = np.abs(np.subtract.outer(mag, mag))
        thr = np.quantile(d[np.isfinite(d)], [0.2, 0.5]) if np.isfinite(d).any() else [0.0, 0.0]
        def fake_diag(dist, t):
            M = (dist < t).astype(np.float32)
            lifetimes = []
            for k in range(-10, 11):
                diag = np.diag(M, k=k)
                run, runs = 0, []
                for v in diag:
                    if v > 0.5: run += 1
                    elif run: runs.append(run); run = 0
                if run: runs.append(run)
                lifetimes.extend(runs)
            if not lifetimes: return np.empty((0,2), dtype=np.float32)
            lifetimes = np.asarray(lifetimes, dtype=np.float32)
            births = np.zeros_like(lifetimes); deaths = lifetimes
            return np.stack([births, deaths], axis=1)
        D0 = fake_diag(d, thr[0]); D1 = fake_diag(d, thr[1])

    feats = []
    for D in (D0, D1):
        if D.size == 0:
            feats.extend([0.0, 0.0, 0.0, 0.0]) # max/mean/sum pers + entropy
            feats.extend([0.0, 0.0, 0.0])     # top3 lifetimes
            feats.extend([0.0, 0.0])          # max birth/death
            feats.extend([0.0, 0.0, 0.0])     # counts > quantiles
            continue
        births, deaths = D[:,0], D[:,1]
        finite = np.isfinite(deaths); births, deaths = births[finite], deaths[finite]
        pers = np.maximum(deaths - births, 0.0)
        feats.append(float(pers.max() if pers.size else 0.0))
        feats.append(float(pers.mean() if pers.size else 0.0))
        feats.append(float(pers.sum() if pers.size else 0.0))
        feats.append(_persistence_entropy(D))
        feats.extend(_topk_lifetimes(D, k=3))
        feats.append(float(births.max() if births.size else 0.0))
        feats.append(float(deaths.max() if deaths.size else 0.0))
        if pers.size >= 5:
            qs = np.quantile(pers, [0.5, 0.75, 0.9])
            feats.extend([float((pers > qs[0]).sum()),
                          float((pers > qs[1]).sum()),
                          float((pers > qs[2]).sum())])
        else:
            feats.extend([0.0, 0.0, 0.0])
    return np.array(feats, dtype=np.float32)

# ===== Dataset wrapper =====
class ArrayDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, subj: np.ndarray, idxs: np.ndarray,
                 patch_len: int, n_patches: int):
        self.X = X[idxs]      # (N,T,C)
        self.y = y[idxs]
        self.subj = subj[idxs]
        self.T = self.X.shape[1]; self.C = self.X.shape[2]
        assert self.T == patch_len * n_patches, f"T={self.T} must equal patch_len* n_patches ({patch_len}*{n_patches})"
        self.patch_len = patch_len
        self.n_patches = n_patches

    def __len__(self): return self.X.shape[0]

    def __getitem__(self, i):
        w = self.X[i]  # (T,C)
        mu = np.mean(w, axis=0, keepdims=True); sd = np.std(w, axis=0, keepdims=True)
        normed = (w - mu) / (sd + 1e-8)
        normed = np.clip(normed, -10, 10).astype(np.float32)

        NP, PL, C = self.n_patches, self.patch_len, self.X.shape[2]
        patches = normed.reshape(NP, PL, C).transpose(2, 0, 1).astype(np.float32)  # (C, NP, PL)

        sfeat = compute_statistical_features(normed, sr=SAMPLING_RATE).astype(np.float32)  # 56
        tfeat = compute_topological_features(normed, sr=SAMPLING_RATE).astype(np.float32)  # 24

        return (
            torch.from_numpy(patches),         # (C, NP, PL)
            torch.from_numpy(sfeat),           # (56,)
            torch.from_numpy(tfeat),           # (24,)
            torch.tensor(self.y[i], dtype=torch.long),
            int(self.subj[i]),
        )

def make_loader(X, y, subj, idxs, patch_len, n_patches, batch=64, shuffle=False):
    ds = ArrayDataset(X, y, subj, idxs, patch_len=patch_len, n_patches=n_patches)
    dl = DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=2, pin_memory=GPU)
    return ds, dl

# ===== RoPE + Model =====
def precompute_freqs_cis(dim: int, n_tokens: int, theta: float = 10000.0):
    assert dim % 2 == 0
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(n_tokens)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

def apply_rotary_pos_emb(q, k, freqs_cis):
    B, H, N, D = q.shape
    d2 = D // 2
    freqs = freqs_cis[:N].to(q.device).view(1, 1, N, d2)
    q_ = q.float().contiguous().view(B, H, N, d2, 2)
    k_ = k.float().contiguous().view(B, H, N, d2, 2)
    q_c = torch.view_as_complex(q_)
    k_c = torch.view_as_complex(k_)
    q_out = torch.view_as_real(q_c * freqs).view(B, H, N, D)
    k_out = torch.view_as_real(k_c * freqs).view(B, H, N, D)
    return q_out.type_as(q), k_out.type_as(k)

class PatchEmbedding(nn.Module):
    """
    Patch tokens + Stats token + Topo token.
    """
    def __init__(self, patch_len=50, channels=9, d_model=56, n_patches=10,
                 n_stat_features=56, n_topo_features=24):
        super().__init__()
        self.n_patches = n_patches
        self.patch_proj = nn.Linear(channels * patch_len, d_model)
        self.stat_proj  = nn.Linear(n_stat_features, d_model)
        self.topo_proj  = nn.Linear(n_topo_features, d_model)
        self.norm       = nn.LayerNorm(d_model)

    def forward(self, patches, stats, topo):
        # patches: (B,C,NP,PL)
        B, C, NP, PL = patches.shape
        assert NP == self.n_patches
        x = patches.permute(0, 2, 1, 3).reshape(B, NP, C * PL)  # (B,NP,C*PL)
        x = self.patch_proj(x)                                  # (B,NP,D)
        s = self.stat_proj(stats).unsqueeze(1)                  # (B,1,D)
        g = self.topo_proj(topo).unsqueeze(1)                   # (B,1,D)
        x = torch.cat([x, s, g], dim=1)                         # (B, NP+2, D)
        return self.norm(x)

class RoPETransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=56, n_heads=2, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim % 2 == 0, "per-head dim must be even for RoPE"
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4*d_model, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, freqs_cis):
        B, N, D = x.shape; H = self.n_heads; d = self.head_dim
        qkv = self.qkv(x).reshape(B, N, 3, H, d).permute(0, 2, 1, 3, 4)
        q = qkv[:, 0].transpose(1, 2); k = qkv[:, 1].transpose(1, 2); v = qkv[:, 2].transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, freqs_cis)
        attn = (q @ k.transpose(-2, -1)) / (d ** 0.5)
        attn = self.attn_drop(attn.softmax(dim=-1))
        out  = (attn @ v).transpose(1, 2).contiguous().reshape(B, N, D)
        x = self.norm1(x + self.drop(self.proj(out)))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x

class RoPETransformerEncoder(nn.Module):
    def __init__(self, d_model=56, n_heads=2, n_layers=2, dropout=0.1, n_tokens=12):
        super().__init__()
        self.layers = nn.ModuleList([RoPETransformerEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        head_dim = d_model // n_heads
        freqs_cis = precompute_freqs_cis(head_dim, n_tokens)
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x, self.freqs_cis)
        return x

class PatchTSTClassifier(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, dropout, n_patches, channels, num_classes, patch_len):
        super().__init__()
        self.embed    = PatchEmbedding(patch_len, channels, d_model, n_patches,
                                       n_stat_features=56, n_topo_features=24)
        self.backbone = RoPETransformerEncoder(d_model, n_heads, n_layers, dropout, n_tokens=n_patches+2)
        self.cls      = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(d_model//2, num_classes),
        )
    def forward(self, patches, stats, topo):
        x = self.embed(patches, stats, topo)  # (B, NP+2, D)
        x = self.backbone(x)                  # (B, NP+2, D)
        x = x.mean(dim=1)                     # (B, D)
        return self.cls(x)                    # (B, K)

# ===== Metrics =====
def cohen_kappa_standard(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n = cm.sum()
    if n == 0: return 0.0
    po = np.trace(cm)/n
    pe = np.dot(cm.sum(1), cm.sum(0))/(n*n)
    return (po - pe) / (1 - pe) if abs(1-pe) > 1e-12 else 0.0

def multiclass_mcc_gorodkin(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred).astype(float)
    n = cm.sum()
    if n == 0: return 0.0
    s = np.trace(cm); t = cm.sum(1); p = cm.sum(0)
    num = s*n - np.sum(t*p)
    den = math.sqrt(max(n**2 - np.sum(t**2), 0.0) * max(n**2 - np.sum(p**2), 0.0))
    return num/den if den > 0 else 0.0

def compute_class_weights(ds, K):
    counts = np.zeros(K, dtype=np.int64)
    for _, s, t, lab, _ in DataLoader(ds, batch_size=512):
        for v in lab.numpy(): counts[v] += 1
    weights = counts.max() / np.clip(counts, 1, None)
    w = torch.tensor(weights, dtype=torch.float32)
    return w / w.sum() * K

def train_one_split(train_dl, val_dl, model, class_w=None, max_epochs=20, patience=5, lr=1e-3, wd=1e-4, max_grad_norm=1.0):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.CrossEntropyLoss(weight=class_w.to(device) if class_w is not None else None)
    scaler = torch.cuda.amp.GradScaler(enabled=GPU)

    best_f1k, best_state, bad = -1e9, None, 0
    for epoch in range(max_epochs):
        # Train
        model.train()
        total = 0.0
        for patches, sfeat, tfeat, labels, _ in train_dl:
            patches = patches.to(device); sfeat = sfeat.to(device); tfeat = tfeat.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with amp_autocast():
                logits = model(patches, sfeat, tfeat)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer); scaler.update()
            total += float(loss.item())
        scheduler.step()

        # Validate
        model.eval()
        preds, truth = [], []
        with torch.no_grad():
            for patches, sfeat, tfeat, labels, _ in val_dl:
                patches = patches.to(device); sfeat = sfeat.to(device); tfeat = tfeat.to(device)
                logits  = model(patches, sfeat, tfeat)
                pred    = logits.argmax(1).cpu().numpy()
                preds.extend(pred.tolist()); truth.extend(labels.numpy().tolist())
        preds = np.array(preds); truth = np.array(truth)
        f1 = f1_score(truth, preds, average="macro")
        k  = cohen_kappa_standard(truth, preds)
        score = f1 + k
        print(f"    Epoch {epoch+1:02d}/{max_epochs}  loss={total/max(1,len(train_dl)):.4f}  F1={f1:.4f}  κ={k:.4f}")
        if score > best_f1k + 1e-6:
            best_f1k = score; best_state = {k:v.cpu() for k,v in model.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("    Early stopping.")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def eval_split(test_dl, model):
    model.eval()
    preds, truth = [], []
    for patches, sfeat, tfeat, labels, _ in test_dl:
        patches = patches.to(device); sfeat = sfeat.to(device); tfeat = tfeat.to(device)
        logits  = model(patches, sfeat, tfeat)
        pred    = logits.argmax(1).cpu().numpy()
        preds.extend(pred.tolist()); truth.extend(labels.numpy().tolist())
    preds = np.array(preds); truth = np.array(truth)
    f1 = f1_score(truth, preds, average="macro")
    k  = cohen_kappa_standard(truth, preds)
    m  = multiclass_mcc_gorodkin(truth, preds)
    return f1, k, m

# ---------------------------
# 4) FIVE-FOLD SUBJECT-WISE CV with 7:1:2 split
# ---------------------------

def subject_fivefold_splits(subject_ids: np.ndarray, seed: int = 42):
    uniq = np.unique(subject_ids)
    rng = np.random.RandomState(seed)
    uniq = uniq.copy()
    rng.shuffle(uniq)

    n = len(uniq)
    n_test = max(1, int(round(0.20 * n)))
    n_val  = max(1, int(round(0.10 * n)))
    folds = []
    for i in range(5):
        start = (i * n_test) % n
        test_idx = [uniq[(start + j) % n] for j in range(n_test)]
        val_start = (start + n_test) % n
        val_idx = [uniq[(val_start + j) % n] for j in range(n_val)]
        train_idx = [s for s in uniq if s not in set(test_idx) and s not in set(val_idx)]
        folds.append((np.array(train_idx), np.array(val_idx), np.array(test_idx)))
    return folds

def mask_from_subjects(all_subjects: np.ndarray, keep_subjects: np.ndarray):
    keep = set(int(s) for s in keep_subjects.tolist())
    return np.array([i for i, s in enumerate(all_subjects) if int(s) in keep], dtype=np.int64)

def run_cv(X, y, subj_ids, patch_len, n_patches, epochs=20, patience=5, lr=1e-3, wd=1e-4, batch=64):
    folds = subject_fivefold_splits(subj_ids, seed=SEED)
    all_metrics = []
    for fold_id, (train_subj, val_subj, test_subj) in enumerate(folds, 1):
        print(f"\n  ── Fold {fold_id}/5 ───────────────────────────────────────────")
        print(f"     subjects: train={len(train_subj)}  val={len(val_subj)}  test={len(test_subj)}")

        idx_train = mask_from_subjects(subj_ids, train_subj)
        idx_val   = mask_from_subjects(subj_ids, val_subj)
        idx_test  = mask_from_subjects(subj_ids, test_subj)

        train_ds, train_dl = make_loader(X, y, subj_ids, idx_train, patch_len, n_patches, batch=batch, shuffle=True)
        val_ds,   val_dl   = make_loader(X, y, subj_ids, idx_val,   patch_len, n_patches, batch=batch, shuffle=False)
        test_ds,  test_dl  = make_loader(X, y, subj_ids, idx_test,  patch_len, n_patches, batch=batch, shuffle=False)

        # Per-class counts (train)
        counts = Counter(train_ds.y.tolist())
        summary = ", ".join([f"{k}:{counts.get(k,0)}" for k in range(num_classes)])
        print(f"     windows: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)} | class_counts(train) [{summary}]")

        channels = X.shape[2]
        model = PatchTSTClassifier(
            d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS, n_layers=cfg.N_LAYERS, dropout=cfg.DROPOUT,
            n_patches=n_patches, channels=channels, num_classes=num_classes, patch_len=patch_len
        ).to(device)

        class_w = compute_class_weights(train_ds, num_classes).to(device)
        model = train_one_split(train_dl, val_dl, model, class_w=class_w,
                                max_epochs=epochs, patience=patience, lr=lr, wd=wd, max_grad_norm=1.0)
        f1, k, m = eval_split(test_dl, model)
        print(f"     TEST: Macro-F1={f1:.4f}  κ={k:.4f}  MCC={m:.4f}")
        all_metrics.append((f1, k, m))

        # Free ASAP
        del model, train_ds, val_ds, test_ds, train_dl, val_dl, test_dl
        if GPU:
            torch.cuda.empty_cache()

    all_metrics = np.array(all_metrics)
    mean = all_metrics.mean(axis=0); std = all_metrics.std(axis=0)
    print("\n  >>> 5-Fold Subject-wise CV (7:1:2 per fold) Summary")
    print(f"      Macro-F1: {mean[0]:.4f} ± {std[0]:.4f}")
    print(f"      Cohen κ : {mean[1]:.4f} ± {std[1]:.4f}")
    print(f"      MCC     : {mean[2]:.4f} ± {std[2]:.4f}")
    return mean, std

# ---------------------------
# 5) RUN FIXED CONFIGURATION
# ---------------------------

def run_fixed_config(
    subjects=None,
    activities=None,
    window_size_sec=10,
    sensors=("acc","gyr","mag"),
    body_part="shin",
    overlap=0.5,
    epochs=20,
    patience=5,
    batch=64,
    lr=1e-3,
    wd=1e-4
):
    if subjects is None:
        subjects = list(range(1, 16))
    if activities is None:
        activities = list(activity_labels.keys())

    print("== Fixed Configuration ==")
    print(f"  Subjects: {subjects}")
    print(f"  Activities: {activities}")
    print(f"  Window size (s): {window_size_sec}")
    print(f"  Sensors: {sensors}")
    print(f"  Body part: {body_part}")
    print(f"  Overlap: {overlap}\n")

    # Fixed configuration: win=10s | patches=10 | patch_len=50 | sensors=acc+gyr+mag | body=shin
    win_sec = 10
    n_patches = 10
    patch_len = 50  # 10 patches of 50 samples each for 10s window (500 samples)

    sensors = ("acc", "gyr", "mag")
    body_part = "shin"
    
    title = f"[win={win_sec:>2}s | patches=10 | patch_len={patch_len} | sensors={'+'.join(sensors)} | body={body_part}]"
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))
    print("  Loading + windowing (may take a bit) ...")
    X, y, subj_ids = generate_all_windows(
        subjects=subjects, activities=activities, window_size_seconds=win_sec,
        quality_check=True, overlap=overlap, use_sensors=sensors, body_parts=[body_part]
    )
    if X.size == 0:
        print("  (No windows produced; skipping this setting.)")
        return

    print(f"  Data shapes: X={X.shape}  y={y.shape}  subj={subj_ids.shape}")
    print(f"  Channels (C) = {X.shape[2]} | Window samples (T) = {X.shape[1]} → 10 patches × {patch_len} samples")

    # Quick class histogram
    cc = Counter(y.tolist())
    hist = ", ".join([f"{classes[k]}:{cc.get(k,0)}" for k in range(num_classes)])
    print(f"  Class histogram: {hist}")

    # Run 5-fold CV
    mean, std = run_cv(
        X, y, subj_ids, patch_len=patch_len, n_patches=n_patches,
        epochs=epochs, patience=patience, lr=lr, wd=wd, batch=batch
    )
    
    print(f"\n  >>> FIXED CONFIGURATION RESULT: win={win_sec}s | patches=10 | patch_len={patch_len} | sensors={'+'.join(sensors)} | body={body_part}")
    print(f"      Macro-F1: {mean[0]:.4f} ± {std[0]:.4f}")
    print(f"      Cohen κ : {mean[1]:.4f} ± {std[1]:.4f}")
    print(f"      MCC     : {mean[2]:.4f} ± {std[2]:.4f}")
    
    # Free memory
    del X, y, subj_ids
    if GPU: torch.cuda.empty_cache()

    return {
        "window_s": win_sec,
        "patches": n_patches,
        "patch_len": patch_len,
        "sensors": "+".join(sensors),
        "body": body_part,
        "mean_macroF1": float(mean[0]),
        "std_macroF1": float(std[0]),
        "mean_kappa": float(mean[1]),
        "std_kappa": float(std[1]),
        "mean_mcc": float(mean[2]),
        "std_mcc": float(std[2]),
    }

# ---------------------------
# 6) Run
# ---------------------------

if __name__ == "__main__":
    result = run_fixed_config(
        subjects=list(range(1, 16)),
        activities=list(activity_labels.keys()),
        window_size_sec=10,
        sensors=("acc","gyr","mag"),
        body_part="shin",
        overlap=0.5,
        epochs=cfg.EPOCHS,
        patience=cfg.EARLY_STOP_PATIENCE,
        batch=cfg.BATCH_SIZE,
        lr=cfg.LR,
        wd=cfg.WEIGHT_DECAY
    )

== Fixed Configuration ==
  Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  Activities: ['climbingdown', 'climbingup', 'jumping', 'lying', 'running', 'sitting', 'standing', 'walking']
  Window size (s): 10
  Sensors: ('acc', 'gyr', 'mag')
  Body part: shin
  Overlap: 0.5


[win=10s | patches=10 | patch_len=50 | sensors=acc+gyr+mag | body=shin]
  Loading + windowing (may take a bit) ...
  Data shapes: X=(11857, 500, 9)  y=(11857,)  subj=(11857,)
  Channels (C) = 9 | Window samples (T) = 500 → 10 patches × 50 samples
  Class histogram: climbingdown:1151, climbingup:1437, jumping:268, lying:1896, running:2066, sitting:1636, standing:1744, walking:1659

  ── Fold 1/5 ───────────────────────────────────────────
     subjects: train=10  val=2  test=3
     windows: train=7977  val=1366  test=2514 | class_counts(train) [0:773, 1:1029, 2:180, 3:1272, 4:1444, 5:1133, 6:1119, 7:1027]
